# Lesson 3.7 — Action Chunking：一次预测一段动作

3.2 建立的误差复利式

$$
e_{t+1}=(1-0.1k)\,e_t
$$

里藏着一个从未被质疑的假设：**策略每一步都重新决策一次**。

但**决策频率是自由的**。本课把"一次预测多少步、执行多少步"变成可以选的量，并测出它的后果。

> **本课核心命题**：chunking 把**高频闭环**换成**低频闭环 + 局部的开环承诺**。它减少的是**重新观察的次数**，不是单次误差 $\varepsilon$。

对应 `docs/roadmap_v3.md` 的 3.7（Action Chunking）。

学完本课后应该能：

1. 写出 chunk 训练样本的精确定义，并断言它**不跨 episode 边界**；
2. 说明 $L/H/K$ 三个超参数各自属于**训练时**还是**执行时**；
3. 测出 per-horizon 误差曲线，并给出**有效视界**；
4. 设计一个**受控**的单步 baseline 对比，并说清"公平比较发生在哪个量上"；
5. 解释为什么 $K$ 能训练后扫描而 $H$ 不能。

**与前后课的关系**

| | 内容 |
|---|---|
| 接 3.2 | 复利式的迭代次数是本课的因变量 |
| 接 3.3 | 离线指标与闭环成功率的分离，在这里再做一次 |
| 接 3.5 / 3.6 | 本课固定 $L=1$，**主动隔离** observation 侧的历史变量 |
| 接 3.4 | ACT 的 chunking 在这里被拆开测量 |
| **本课不做** | Diffusion / Flow Matching、temporal ensembling 的实现、数据扩容 |

## 1. 问题：决策频率是一个可以选的东西

把 policy 从

$$
a_t=\pi(o_t)
$$

改成"**一次预测 $H$ 步，但只执行 $K$ 步，然后重新观察**"：

$$
[\hat a_t,\dots,\hat a_{t+H-1}]=\pi(o_t),
\qquad
\text{执行 } \hat a_t,\dots,\hat a_{t+K-1},
\qquad
\text{再从 } o_{t+K} \text{ 重新预测}
$$

它带来三个可以直接测量的问题：

1. **越远的动作越难预测吗？**（离线，per-horizon 曲线）
2. **把决策点从 $T$ 降到 $T/K$，闭环会变好吗？**（在线，$K$ 扫描）
3. **$K$ 变大换来了什么、又付出了什么？**

第 3 问的答案是本课的落点：**决策次数下降，但一次预测要盲走的区间（开环区间）上升**。

## 2. 符号：$L$ / $H$ / $K$

| 符号 | 含义 | 属于 | 本课取值 |
|---|---|---|---|
| $L$ | 输入多少帧 observation | **训练时**（输入形状） | 1（隔离变量） |
| $H$ | 一次预测多少步 action | **训练时**（输出形状） | 8（= 0.40 s @ 20 Hz） |
| $K$ | 预测后先执行几步再重规划 | **执行时**（策略超参） | 扫 $\{1,2,4,8\}$ |

**$H$ 和 $K$ 出现在完全不同的地方，这是本课最容易混的一点：**

- $H$ 决定**目标有多长**，因此决定每个样本长什么样、以及丢多少起点；
- $K$ 决定**执行时用掉预测的前几行**，只出现在 rollout 里，**不进数据集**。

所以下面的 chunk 构造代码里你**找不到 $K$**。也正因为如此：

> **训练时起点每次 +1，执行时起点每次 +K。**
> 一次训练就能扫 $K$，因为 $K$ 没有被写进数据。而改 $H$ 必须重训。

（$H-K$ 是相邻两次预测的**重叠长度**：$K=H$ 时重叠为 0，不能做 temporal ensembling；$K=1$ 时重叠最大。）

In [1]:
# Cell 1 - pin down the training-sample contract before building any sample
import h5py
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPERT_H5 = PROJECT_ROOT / "datasets" / "pickcube" / "expert_episodes.h5"
CONTROL_MODE = "pd_joint_pos"            # must match the file, not the rollout default
OBS_DIM, ACTION_DIM = 42, 8

with h5py.File(EXPERT_H5, "r") as f:
    assert f.attrs["control_mode"] == CONTROL_MODE, f.attrs["control_mode"]
    assert f.attrs["transition_schema_version"] == 2
    episodes = []
    for name in sorted(f.keys()):
        observations = f[name]["observations"][:]   # [T+1, 42] : o_0 .. o_T
        actions      = f[name]["actions"][:]        # [T,   8]  : a_0 .. a_{T-1}
        # Canonical schema: T+1 states, T actions, o_t is the state *before* a_t.
        # Equal length cannot prove the pairing, so assert it instead of assuming it.
        assert observations.shape[0] == actions.shape[0] + 1, name
        assert observations.shape[1] == OBS_DIM
        assert actions.shape[1] == ACTION_DIM
        episodes.append({
            "name": name,
            "state": observations[:-1],   # policy view: o_0 .. o_{T-1}
            "action": actions,            #               a_0 .. a_{T-1}
            "next_state": observations[1:],
        })

print(f"{len(episodes)} episodes | control_mode={CONTROL_MODE}")
print("T per episode    :", [len(e["action"]) for e in episodes])
print("total (o_t, a_t) :", sum(len(e["action"]) for e in episodes))
print("state  :", episodes[0]["state"].dtype, episodes[0]["state"].shape)
print("action :", episodes[0]["action"].dtype, episodes[0]["action"].shape)

5 episodes | control_mode=pd_joint_pos
T per episode    : [74, 74, 50, 86, 76]
total (o_t, a_t) : 360
state  : float32 (74, 42)
action : float32 (74, 8)


### 上面那个 cell 在检查什么

一个 episode 其实是**两张表**（以 `episode_000000` 为例）：

| 表 | shape | 读作 |
|---|---|---|
| `observations` | `(75, 42)` | 75 行、每行 42 个数 |
| `actions` | `(74, 8)` | 74 行、每行 8 个数 |

观测比动作**多一行**，因为文件存的是"**做动作之前**的状态"：

```text
 o_0 --a_0--> o_1 --a_1--> ... --> o_73 --a_73--> o_74
 ^                                                 ^
还没做任何动作                                  做完最后一个动作
```

- 状态（圆点）75 个：$o_0 \dots o_{74}$；动作（箭头）74 个：$a_0 \dots a_{73}$
- 多出来的 $o_{74}$ **没有配对的动作**，是终点标记，训练时用不到

所以 policy 用的部分是 `state = observations[:-1]`，得到 `(74, 42)` 与 `(74, 8)`，**行号一一对应**。

**为什么会有那个 +1 ？** 因为"从 $a$ 数到 $b$（含两端）一共有 $b-a+1$ 个整数"。同一条规则解释了本课的三个量：

| 数什么 | 从 | 到 | 个数 |
|---|---|---|---|
| 观测行数 | 0 | $T$ | $T+1$ |
| 动作行数 | 0 | $T-1$ | $T$ |
| chunk 起点 | 0 | $T-H$ | $T-H+1$ |

**assert 是什么。** 它是你写给计算机的一句检查话："我断定这件事成立；**不成立就立刻停下报错**。"

为什么需要它？因为**写错的时候计算机不会自动报错**。它会继续往下跑，然后给你一条看起来很正常的 loss 曲线。这个仓库就吃过这个亏：Lesson 2 有一版文件存的是动作**之后**的观测，两个数组行数一样，形状检查抓不到。所以 `assert` 不是装饰，是**唯一能在数据层就拦下错误的东西**。

In [2]:
# Cell 2 - chunk samples, built strictly inside each episode
H = 8            # chunk length: a_t .. a_{t+H-1}, exactly H actions
CONTROL_HZ = 20

def build_chunk_dataset(episodes, H):
    """Return (states, chunks, episode_of, start_of).

    states[i]     -> o_t
    chunks[i]     -> [a_t, ..., a_{t+H-1}], all from ONE episode
    episode_of[i] -> which episode, so the split can stay episode-level
    start_of[i]   -> t, for auditing and for temporal ensembling later
    """
    states, chunks, episode_of, start_of = [], [], [], []
    for ei, ep in enumerate(episodes):
        states_t, actions_t = ep["state"], ep["action"]
        n = len(actions_t)
        assert n - H + 1 > 0, f"{ep['name']}: T={n} is shorter than H={H}"
        for t in range(n - H + 1):        # the last H-1 steps are dropped
            states.append(states_t[t])
            chunks.append(actions_t[t:t + H])
            episode_of.append(ei)
            start_of.append(t)
    return (np.stack(states), np.stack(chunks),
            np.asarray(episode_of), np.asarray(start_of))

states, chunks, episode_of, start_of = build_chunk_dataset(episodes, H)
print("states:", states.shape, "| chunks:", chunks.shape, f"(H={H})")
print("samples per episode:", [int((episode_of == i).sum()) for i in range(len(episodes))])

# Assert the property the loop exists to guarantee.
assert all(start_of[i] + H <= len(episodes[episode_of[i]]["action"])
           for i in range(len(chunks)))
assert all(np.array_equal(chunks[i],
               episodes[episode_of[i]]["action"][start_of[i]:start_of[i] + H])
           for i in range(len(chunks)))
print("no chunk crosses an episode boundary: OK")
print(f"one chunk = {H} steps = {H / CONTROL_HZ:.2f} s of future motion")

states: (325, 42) | chunks: (325, 8, 8) (H=8)
samples per episode: [67, 67, 43, 79, 69]
no chunk crosses an episode boundary: OK
one chunk = 8 steps = 0.40 s of future motion


### 为什么必须一条一条 episode 地做，以及丢掉了什么

**为什么不能先把所有 episode 的动作拼成一个大数组再切？** 因为那样末尾的 chunk 会**读到下一条 episode 的动作**。下一个 cell 把这件事算出来。

**丢掉的是哪些起点？** 起点 $t$ 必须满足"往后够 $H$ 行"：

$$
t+H\le T \quad\Longrightarrow\quad t\le T-H
$$

所以可用的 $t$ 是 $0,1,\dots,T-H$，一共 $T-H+1$ 个。每条 episode 用不了的起点是 $t=T-H+1,\dots,T-1$，**正好 $H-1$ 个**：

$$\text{样本数}=\sum_{i=1}^{N}\bigl(T_i-H+1\bigr)
=\sum_{i=1}^{N}T_i \;-\; N\,(H-1)$$

代入本题（$\sum T_i=360$，$N=5$，$H=8$）：

$$360-5\times(8-1)=360-35=\mathbf{325}$$

| $H$ | 逐条 $T-H+1$ | 样本数 | 比单步少 |
|---:|---|---:|---:|
| 1（单步） | [74, 74, 50, 86, 76] | 360 | 0 |
| 2 | [73, 73, 49, 85, 75] | 355 | 5 |
| 4 | [71, 71, 47, 83, 73] | 345 | 15 |
| **8** | [67, 67, 43, 79, 69] | **325** | **35** |
| 16 | [59, 59, 35, 71, 61] | 285 | 75 |

**注意丢掉的 35 个起点是每条 episode 的结尾**，也就是"把方块放下"那几步。这是一个真实代价，不是无害的截断。

**$H$ 的上限**由**最短那条** episode 决定：$\min_i T_i=50$，所以 $H\le 50$。$H=51$ 时 T=50 那条会贡献 **0** 个样本；$H=52$ 时出现**负数**——这正是上面那个 `assert n - H + 1 > 0` 拦的东西。

最后记一句：样本数由 $H$ 决定这件事**与 $K$ 无关**。$K$ 取 1、2、4 还是 8，上面这张表一个数字都不变。

In [3]:
# 反例：先把所有 episode 拼起来再切，会静默地跨过 episode 边界
naive_actions = np.concatenate([ep["action"] for ep in episodes])
bounds = np.cumsum([len(ep["action"]) for ep in episodes])[:-1]

naive_samples = len(naive_actions) - H + 1
crossing = sum(any(t < b < t + H for b in bounds) for t in range(naive_samples))

print(f"episode-local (correct) : {len(chunks):>4} samples, shape {chunks.shape}")
print(f"naive concat-then-slice : {naive_samples:>4} samples, shape "
      f"{np.stack([naive_actions[t:t + H] for t in range(naive_samples)]).shape}")
print(f"naive samples spanning TWO episodes: {crossing}")
print()
print(f"crossing == boundaries * (H-1) : {crossing} == {len(bounds)} * {H - 1} "
      f"= {len(bounds) * (H - 1)}")
assert crossing == len(bounds) * (H - 1)
assert naive_samples == len(chunks) + crossing

# Both arrays are valid tensors of the same rank. Only the episode-local loop
# guarantees that a chunk never mixes two trajectories.
print("both arrays are valid; a shape check cannot catch this")


episode-local (correct) :  325 samples, shape (325, 8, 8)
naive concat-then-slice :  353 samples, shape (353, 8, 8)
naive samples spanning TWO episodes: 28

crossing == boundaries * (H-1) : 28 == 4 * 7 = 28
both arrays are valid; a shape check cannot catch this


上面这个 cell 是本课唯一真正的"坑"：两种做法**都产出合法的数组**，`np.stack` 不报错、loss 照常下降，但 naive 版本里有 **28 个样本把两条不同 episode 的动作拼成了一个 chunk**。

差值还有两个漂亮的自洽关系，都被 `assert` 固定住了：

```text
crossing      = (episode 条数 - 1) x (H - 1) = 4 x 7 = 28
naive 样本数  = 正确样本数 + crossing        = 325 + 28 = 353
```

以后改 $H$ 或换数据集时，这两个等式会立刻告诉你有没有越界。

In [4]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Episode-level split. order[0] -- NOT order[-1] -- is the held-out episode.
# default_rng(42).permutation(5) == [4 2 3 1 0], so order[0] == 4, which is the
# held-out episode Lesson 2 recorded: train 000000..000003, validation 000004.
order = np.random.default_rng(42).permutation(len(episodes))
train_ids, val_ids = order[1:], order[:1]
assert set(val_ids.tolist()) == {4}, f"held-out must be episode_000004, got {val_ids}"

train_mask = np.isin(episode_of, train_ids)
val_mask = np.isin(episode_of, val_ids)

# Normalisation statistics come from the TRAINING episodes' raw frames -- never
# from the overlapping chunks, never from the validation episode.
train_states_raw = np.concatenate([episodes[i]["state"] for i in train_ids], axis=0)
train_actions_raw = np.concatenate([episodes[i]["action"] for i in train_ids], axis=0)

state_mean = train_states_raw.mean(axis=0)
state_std = np.maximum(train_states_raw.std(axis=0), 1e-6)
action_mean = train_actions_raw.mean(axis=0)
action_std = np.maximum(train_actions_raw.std(axis=0), 1e-6)

X_train = ((states[train_mask] - state_mean) / state_std).astype(np.float32)
Y_train = ((chunks[train_mask] - action_mean) / action_std).astype(np.float32)
X_val = ((states[val_mask] - state_mean) / state_std).astype(np.float32)
Y_val = ((chunks[val_mask] - action_mean) / action_std).astype(np.float32)

chunk_train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train)),
    batch_size=8, shuffle=True)
chunk_val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val), torch.from_numpy(Y_val)),
    batch_size=8, shuffle=False)

# The single-step target for sample i is the chunk's FIRST row, a_t. Sharing X
# keeps the sample set identical by construction. Keep the horizon dimension
# (0:1, not 0): the model returns [B, 1, 8], and F.mse_loss would otherwise
# SILENTLY BROADCAST [B, 1, 8] against [B, 8] into a [B, B, 8] garbage loss.
single_train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train[:, 0:1, :])),
    batch_size=8, shuffle=True)
single_val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val), torch.from_numpy(Y_val[:, 0:1, :])),
    batch_size=8, shuffle=False)

print("train episodes:", sorted(train_ids.tolist()), "| val episode:", val_ids.tolist())
print("train chunks:", tuple(X_train.shape), "| val chunks:", tuple(Y_val.shape))
print("raw train frames used for normalisation:", train_states_raw.shape, train_actions_raw.shape)


train episodes: [0, 1, 2, 3] | val episode: [4]
train chunks: (256, 42) | val chunks: (69, 8, 8)
raw train frames used for normalisation: (284, 42) (284, 8)


### 为什么 split 必须按 episode，以及为什么 Lesson 2 的数字一个都不能借用

同一 episode 的相邻帧高度相关，按帧划分会把近重复样本放到两边（3.1 已实测：专家轨迹的 consecutive/random 距离比是 `8.8x-13.7x`，随机 fixture 只有 `1.04x`）。

更要紧的是：**Lesson 2 的数字（`0.2350` / `0.1421` / `0/10`）在本课不能直接比较**，有三个独立原因：

| # | Lesson 2 | 本课 | 后果 |
|---:|---|---|---|
| 1 | val = `episode_000004` | 本课也是 `episode_000004`（已用 `order[0]` 对齐） | 这一条已修好 |
| 2 | **只归一化 observation**，action 保持**原始空间** | **action 也归一化** | **误差读数在两个空间** |
| 3 | hidden 64 / batch 32 / epochs 300 / patience 40 | hidden 128 / batch 8 / epochs 150 / patience 25 | 配置不同 |

第 2 条是根本原因。同一个"什么都不学"的误差，在两个空间里的读数是：

```text
原始 action 空间        : 0.14210252   <- Lesson 2 记录的那个数（已精确复现）
本课归一化 action 空间   : 0.7874
```

> 顺带为本仓库正名：Lesson 2 自己那次 `0.2350` vs `0.1421` 的比较**是成立的**，它在原始空间里被精确复现到小数点后 8 位。问题只在于它**不能跨到本课**。

所以本课必须在**自己的配置下**重跑一个受控的单步 baseline。

**另一个同一类的坑（这个 notebook 真的踩到了）。** `F.mse_loss` **会广播**。`ChunkMLP` 即使 `horizon=1` 也返回 `[B, 1, 8]`，如果单步 loader 给的是 `[B, 8]`，两者会被广播成 `[B, B, 8]`，得到一个**看起来正常、其实完全错误**的 loss，梯度也是错的——单步模型会表现为"train loss 卡住不动"。

所以：单步 loader 用 `Y[:, 0:1, :]`（**保留 horizon 那一维**），并且在算 loss 之前 `assert pred.shape == yb.shape`。

> 这跟 §3 那个 episode 边界是同一类错误：**形状"能广播"或"同 rank"，所以没有任何东西会报错。** 唯一可靠的做法是显式断言。

In [5]:
import torch
from torch import nn

torch.manual_seed(42)
H = chunks.shape[1]              # 8
ACTION_DIM = chunks.shape[2]     # 8


class ChunkMLP(nn.Module):
    # Predict `horizon` consecutive actions from one observation.
    def __init__(self, horizon, action_dim, hidden=128):
        super().__init__()
        self.horizon = horizon
        self.action_dim = action_dim
        self.net = nn.Sequential(
            nn.Linear(42, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, horizon * action_dim),
        )

    def forward(self, state):
        flat = self.net(state)
        return flat.reshape(-1, self.horizon, self.action_dim)


def count_params(m):
    return sum(p.numel() for p in m.parameters())


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ChunkMLP(H, ACTION_DIM).to(device)

x, y = next(iter(chunk_train_loader))
pred = model(x.to(device))
print("device:", device)
print("input: ", tuple(x.shape))
print("output:", tuple(pred.shape))
print("target:", tuple(y.shape))
assert pred.shape == y.shape

print()
print("参数量从哪来：最后一层是 Linear(hidden, H*action_dim)")
for h in (1, 2, 4, 8):
    print(f"  H={h:>2} hidden=128 : {count_params(ChunkMLP(h, ACTION_DIM)):>7,} params")
print(f"  H= 1 hidden=150 : {count_params(ChunkMLP(1, ACTION_DIM, hidden=150)):>7,} params"
      "   <- 与 H=8/hidden=128 参数量匹配的单步基线")
print()
print(f"  chunked H={H} hidden=128 : {count_params(model):,} params")


device: cpu
input:  (8, 42)
output: (8, 8, 8)
target: (8, 8, 8)

参数量从哪来：最后一层是 Linear(hidden, H*action_dim)
  H= 1 hidden=128 :  23,048 params
  H= 2 hidden=128 :  24,080 params
  H= 4 hidden=128 :  26,144 params
  H= 8 hidden=128 :  30,272 params
  H= 1 hidden=150 :  30,308 params   <- 与 H=8/hidden=128 参数量匹配的单步基线

  chunked H=8 hidden=128 : 30,272 params


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


### $H$ 改了参数量——这是最容易让整个对比失效的事实

最后一层是 `Linear(hidden, H x 8)`，所以**输出形状一变，参数量就变**：

| 层 | 计算 | $H=1$ | $H=8$ |
|---|---|---:|---:|
| `Linear(42,128)` | $42\times128+128$ | 5,504 | 5,504 |
| `Linear(128,128)` | $128\times128+128$ | 16,512 | 16,512 |
| `Linear(128,H\times8)` | $128\cdot 8H+8H$ | 1,032 | 8,256 |
| **合计** | | **23,048** | **30,272** |

$$\text{chunked 比单步多 } 30{,}272-23{,}048=7{,}224 \;(+31.3\%)$$

**所以**：如果 chunked 的验证误差更低，它**可能只是因为多了 31% 的参数**。要排除这个解释，必须再加一个**参数量匹配**的单步模型（上面打印的 `hidden=150`，30,308 个参数）。

In [6]:
import copy
import torch.nn.functional as F

CONFIG = dict(hidden=128, batch_size=8, max_epochs=150, patience=25, lr=1e-3, seed=42)


def train_model(m, train_loader, val_loader, config=CONFIG, verbose=True):
    # One training code path, shared by every model in this lesson.
    torch.manual_seed(config["seed"])
    optimizer = torch.optim.Adam(m.parameters(), lr=config["lr"])
    best = dict(val=float("inf"), epoch=-1, state=None)
    history, stale = [], 0

    for epoch in range(1, config["max_epochs"] + 1):
        m.train()
        tr_sum = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = m(xb)
            assert pred.shape == yb.shape, (pred.shape, yb.shape)
            loss = F.mse_loss(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            tr_sum += loss.item() * len(xb)
        tr = tr_sum / len(train_loader.dataset)

        m.eval()
        va_sum = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = m(xb)
                assert pred.shape == yb.shape, (pred.shape, yb.shape)
                va_sum += F.mse_loss(pred, yb).item() * len(xb)
        va = va_sum / len(val_loader.dataset)
        history.append((epoch, tr, va))

        if va < best["val"]:
            best = dict(val=va, epoch=epoch, state=copy.deepcopy(m.state_dict()))
            stale = 0
        else:
            stale += 1

        if verbose and (epoch == 1 or epoch % 25 == 0):
            print(f"  epoch {epoch:3d} | train {tr:.6f} | val {va:.6f}")
        if stale >= config["patience"]:
            if verbose:
                print(f"  early stop at epoch {epoch}")
            break

    m.load_state_dict(best["state"])
    m.eval()
    return dict(best_val=best["val"], best_epoch=best["epoch"], history=history)


print(f"CONFIG = {CONFIG}")
print(f"chunked: H={H}, params={count_params(model):,}")
chunked_result = train_model(model, chunk_train_loader, chunk_val_loader)
print(f"chunked best val MSE = {chunked_result['best_val']:.6f} "
      f"at epoch {chunked_result['best_epoch']}")
print()
print("注意：这是 [B, H, 8] 全部 64 个元素上的平均，是本课最难的量，")
print("      不能拿去和单步模型（8 个元素）的 MSE 直接比。")


CONFIG = {'hidden': 128, 'batch_size': 8, 'max_epochs': 150, 'patience': 25, 'lr': 0.001, 'seed': 42}
chunked: H=8, params=30,272


  epoch   1 | train 0.710048 | val 1.132059


  epoch  25 | train 0.009602 | val 0.922903


  early stop at epoch 37
chunked best val MSE = 0.853913 at epoch 12

注意：这是 [B, H, 8] 全部 64 个元素上的平均，是本课最难的量，
      不能拿去和单步模型（8 个元素）的 MSE 直接比。


### 受控 = 只改一个变量

`train_model` 是全课**唯一**的训练代码路径，三个模型共用。受控体现在：

| 项 | 值 | 说明 |
|---|---|---|
| split | train = episodes 0-3，val = episode 4 | 三个模型**引用同一批变量** |
| 归一化 | 训练 episode 的原始帧 | 三个模型**引用同一个 `action_mean/action_std`** |
| hidden | 128（B2 除外，取 150） | 见上一节：B2 是为了匹配参数量 |
| batch / epochs / patience / lr / seed | 8 / 150 / 25 / 1e-3 / 42 | 完全一致 |
| **唯一变化** | **输出形状：8 vs $H\times8$** | |

In [7]:
# Controlled single-step baselines. Same split, same normalisation, same
# hyper-parameters, same seed; the ONLY difference is the output shape.
# B1: same nominal config (hidden=128).  B2: parameter-matched (hidden=150).
baselines = {}
for name, hidden in (("B1", 128), ("B2", 150)):
    torch.manual_seed(CONFIG["seed"])
    m = ChunkMLP(horizon=1, action_dim=ACTION_DIM, hidden=hidden).to(device)
    print(f"--- {name}: single-step, hidden={hidden}, params={count_params(m):,}")
    res = train_model(m, single_train_loader, single_val_loader)
    baselines[name] = dict(model=m, hidden=hidden, params=count_params(m), **res)
    print(f"{name} best val MSE = {res['best_val']:.6f} at epoch {res['best_epoch']}")

print()
for name, b in baselines.items():
    print(f"  {name}: params={b['params']:,}  best val MSE={b['best_val']:.6f}")
print(f"  chunked: params={count_params(model):,}  best val MSE={chunked_result['best_val']:.6f}")


--- B1: single-step, hidden=128, params=23,048
  epoch   1 | train 0.549148 | val 0.752115


  epoch  25 | train 0.006927 | val 0.657297


  early stop at epoch 35
B1 best val MSE = 0.575204 at epoch 10
--- B2: single-step, hidden=150, params=30,308
  epoch   1 | train 0.460125 | val 0.634114


  epoch  25 | train 0.007021 | val 0.563711


  epoch  50 | train 0.005018 | val 0.499332
  early stop at epoch 55
B2 best val MSE = 0.427904 at epoch 30

  B1: params=23,048  best val MSE=0.575204
  B2: params=30,308  best val MSE=0.427904
  chunked: params=30,272  best val MSE=0.853913


### 公平比较发生在**哪一个量**上

这里有一个必须讲清的陷阱：

- 单步模型的验证 MSE 是 **8 个数**上的平均；
- chunked 模型的验证 MSE 是 **64 个数**上的平均，其中 $h=1\dots7$ 天生更难。

**所以这两个总 MSE 不可比**——chunked 被要求做一件严格更难的事，直接比会得出"chunked 更差"的假结论。

**唯一公平的比较量是 $h=0$**：chunked 模型预测"下一步"的那 8 个数，与单步模型预测的那 8 个数，是同一个任务、同一批样本、同一个归一化空间。这个量要在下一节的 per-horizon 诊断里才拿得到，所以最终对比表放在那里。

In [8]:
def eval_mse_per_horizon(m, loader, horizon):
    # Per-horizon MSE. Every model and every loader in this lesson yields
    # [B, horizon, 8], so a shape mismatch is an error, not something to broadcast.
    err = torch.zeros(horizon, device=device)
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            p = m(xb)
            assert p.shape == yb.shape, (p.shape, yb.shape)
            err += (p - yb).square().sum(dim=(0, 2))
            n += len(xb)
    return (err / (n * ACTION_DIM)).cpu().numpy()


mse_chunked = eval_mse_per_horizon(model, chunk_val_loader, H)
mse_b1 = eval_mse_per_horizon(baselines["B1"]["model"], single_val_loader, 1)
mse_b2 = eval_mse_per_horizon(baselines["B2"]["model"], single_val_loader, 1)

# "Predict the training mean" == predict 0 in normalised space.
# Each horizon needs ITS OWN baseline -- see the note below.
baseline_h = (Y_val ** 2).mean(axis=(0, 2))
baseline_0 = float(baseline_h[0])

# Relative skill over the matched baseline. S_h > 0 means "better than predicting
# the training-mean action at this horizon".
S_h = 1.0 - mse_chunked / baseline_h

print(f"{'h':>2} | {'chunked mse':>12} | {'its own baseline':>17} | {'S_h':>8} | better?")
print("-" * 68)
for h in range(H):
    print(f"{h:>2} | {mse_chunked[h]:>12.6f} | {baseline_h[h]:>17.6f} | {S_h[h]:>8.4f} | "
          f"{bool(S_h[h] > 0)}")

# Useful horizon = how many CONSECUTIVE horizons from h=0 beat their own baseline.
# A far horizon that happens to turn positive must not be allowed to skip the
# failures in between. The threshold is fixed here, before looking at the numbers.
USEFUL_THRESHOLD = 0.0
useful = 0
for h in range(H):
    if S_h[h] > USEFUL_THRESHOLD:
        useful += 1
    else:
        break
print()
print(f"useful horizon (consecutive from h=0, threshold {USEFUL_THRESHOLD}) = {useful} of H={H}")
print(f"  covers {useful / CONTROL_HZ:.2f} s of committed motion at {CONTROL_HZ} Hz")

# This identity holds ONLY because drop-tail gives every h the same element count.
assert np.isclose(mse_chunked.mean(), chunked_result["best_val"]), \
    (mse_chunked.mean(), chunked_result["best_val"])
print(f"mean(mse_per_horizon) == best val loss : {mse_chunked.mean():.6f}  (asserted)")

print()
print("=== 公平对比：所有模型都在 h=0 这同一个任务上 ===")
print(f"{'model':>36} | {'params':>7} | {'val MSE @ h=0':>13} | {'overall val MSE':>15}")
print("-" * 84)
print(f"{'mean-action baseline (predict 0)':>36} | {0:>7} | {baseline_0:>13.6f} | {'-':>15}")
for name, b, mse in (("B1 single-step hidden=128", baselines["B1"], mse_b1),
                     ("B2 single-step hidden=150 (matched)", baselines["B2"], mse_b2)):
    print(f"{name:>36} | {b['params']:>7,} | {mse[0]:>13.6f} | {mse[0]:>15.6f}")
print(f"{'chunked H=8 hidden=128':>36} | {count_params(model):>7,} | "
      f"{mse_chunked[0]:>13.6f} | {mse_chunked.mean():>15.6f}")


 h |  chunked mse |  its own baseline |      S_h | better?
--------------------------------------------------------------------
 0 |     0.571828 |          0.557645 |  -0.0254 | False
 1 |     0.765444 |          0.593502 |  -0.2897 | False
 2 |     0.775566 |          0.631174 |  -0.2288 | False
 3 |     0.806682 |          0.670458 |  -0.2032 | False
 4 |     0.872954 |          0.711120 |  -0.2276 | False
 5 |     0.937172 |          0.752895 |  -0.2448 | False
 6 |     1.065003 |          0.795489 |  -0.3388 | False
 7 |     1.036653 |          0.838598 |  -0.2362 | False

useful horizon (consecutive from h=0, threshold 0.0) = 0 of H=8
  covers 0.00 s of committed motion at 20 Hz
mean(mse_per_horizon) == best val loss : 0.853913  (asserted)

=== 公平对比：所有模型都在 h=0 这同一个任务上 ===
                               model |  params | val MSE @ h=0 | overall val MSE
------------------------------------------------------------------------------------
    mean-action baseline (predict 0) |       

### 陷阱：`baseline_h` **自己**随 $h$ 上升

"什么都不学"的 baseline 在 val（`episode_000004`，本课归一化空间）上实测：

| $h$ | `baseline_h` | 覆盖的起点 $t$ |
|---:|---:|---|
| 0 | **0.5576** | 0 … 66 |
| 1 | 0.5935 | 1 … 67 |
| 2 | 0.6312 | 2 … 68 |
| 3 | 0.6705 | 3 … 69 |
| 4 | 0.7111 | 4 … 70 |
| 5 | 0.7529 | 5 … 71 |
| 6 | 0.7955 | 6 … 72 |
| 7 | **0.8386** | 7 … 73 |

极差 **0.281**，涨了 50%。原因有两个，**必须分开**：

1. 更远的动作**确实更难**预测；
2. **采样窗口在移动**：$h=0$ 覆盖 $t\in[0,66]$，$h=7$ 覆盖 $t\in[7,73]$，$h$ 越大越包含 episode 的**尾段**，而尾段的动作偏离均值更多。

所以正确的读法是：

> 在每个 $h$ 上比较 $\text{mse}_h$ 与**它自己的** $\text{baseline}_h$，**而不是**用"$\text{mse}_h$ 随 $h$ 上升"去论证"越远越难"。

由此得到本课最有用的一个数字：

先用相对分数把"比 baseline 好多少"写下来：

$$S_h=1-\frac{\text{mse}_h}{\text{baseline}_h}
\qquad (S_h>0 \iff \text{该步优于 baseline})$$

然后——**这里必须用"从 $h=0$ 起连续"的定义**，而不是"最强的那个 $h$"：

$$\boxed{\text{有效视界}=\text{从 } h=0 \text{ 起连续满足 } S_h>0 \text{ 的步数}}$$

为什么不能用 $\max\{h:\text{mse}_h<\text{baseline}_h\}+1$？因为那样**某个较远的孤立步偶然转正，就会把中间全部失败的步一起算进来**，得到一个虚高的数字。

它回答：**模型比"什么都不学"更强的连续步数是多少？** 有效视界若为 $m$、控制频率 20 Hz，就覆盖约 $m/20$ 秒的"敢盲走"区间。如果有效视界只有 4，那 $H=8$ 就有 4 步是白算的——这是"$H$ 该取多大"的第一个真实线索。

阈值也可以设得比 0 更严格（比如要求 $S_h>0.05$），但**必须在看到数字之前就固定**，否则就是事后调参。代码里它就是 `USEFUL_THRESHOLD`。

（顺带解释前面两组数字的差别：丢尾巴后 val 只剩 69 个 chunk 样本，而尾段偏偏是最"偏"的帧，所以 `0.6939`（69 样本）比 `0.7874`（76 帧）小。这本身也印证了第 2 个原因。）

## 9. $H$ 扫描：$H=8$ 是不是选错了？

有效视界 = 0 时一个自然的怀疑是：**是不是 $H$ 选错了？** 本课一开始把 $H=8$ 当既定值（沿用草稿里的设定），从没验证过。

这一节把它扫一遍，但有一条必须守住的规矩：

> **所有 $H$ 必须共享同一个样本集和同一个 split。**

因为样本数本身就依赖 $H$（丢尾巴 $\sum T_i - N(H-1)$）。如果每个 $H$ 用自己的自然样本集，那么"$H$ 的影响"和"样本数的影响"就混在一起了。做法很自然：$H$ 较小的目标就是 $H=8$ 目标的**前缀**（`Y[:, :H, :]`），共享起点集后唯一变化的就是目标长度。

$H=1$ 那一行**应该精确等于前面的 B1**（同 hidden、同 seed、同样本）——这是一个免费的自我一致性检查。

In [9]:
H_SWEEP = (1, 2, 4, 8)
sweep = {}

for hs in H_SWEEP:
    torch.manual_seed(CONFIG["seed"])
    m = ChunkMLP(hs, ACTION_DIM, hidden=CONFIG["hidden"]).to(device)
    # Nested design: the H-step target is the first H columns of the H=8 target, so
    # every H trains and validates on exactly the same starts.
    tr = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train[:, :hs, :])),
                    batch_size=8, shuffle=True)
    va = DataLoader(TensorDataset(torch.from_numpy(X_val), torch.from_numpy(Y_val[:, :hs, :])),
                    batch_size=8, shuffle=False)
    res = train_model(m, tr, va, verbose=False)

    mse_h = eval_mse_per_horizon(m, va, hs)
    S = 1.0 - mse_h / baseline_h[:hs]        # same baseline_h: same validation samples
    useful = 0
    for k in range(hs):
        if S[k] > USEFUL_THRESHOLD:
            useful += 1
        else:
            break
    sweep[hs] = dict(model=m, params=count_params(m), mse_h=mse_h, S=S,
                     useful=useful, **res)

print(f"{'H':>2} | {'params':>7} | {'overall val MSE':>15} | {'mse @ h=0':>10} | "
      f"{'S @ h=0':>8} | useful")
print("-" * 68)
for hs in H_SWEEP:
    s = sweep[hs]
    print(f"{hs:>2} | {s['params']:>7,} | {s['best_val']:>15.6f} | {s['mse_h'][0]:>10.6f} | "
          f"{s['S'][0]:>8.4f} | {s['useful']}")

print()
print("S_h matrix (rows = H, columns = h); '-' where that H has no such horizon")
print("     " + "".join(f"{('h=' + str(h)):>10}" for h in range(8)))
for hs in H_SWEEP:
    print(f"H={hs:>2} " + "".join(
        f"{sweep[hs]['S'][h]:>10.4f}" if h < hs else f"{'-':>10}" for h in range(8)))

b1 = baselines["B1"]["best_val"]
assert abs(sweep[1]["best_val"] - b1) < 1e-9, (sweep[1]["best_val"], b1)
print()
print(f"self-consistency check: H=1 == B1 exactly ({b1:.6f}) -> asserted")

 H |  params | overall val MSE |  mse @ h=0 |  S @ h=0 | useful
--------------------------------------------------------------------
 1 |  23,048 |        0.575204 |   0.575204 |  -0.0315 | 0
 2 |  24,080 |        0.602890 |   0.638865 |  -0.1456 | 0
 4 |  26,144 |        0.613326 |   0.451150 |   0.1910 | 1
 8 |  30,272 |        0.853913 |   0.571828 |  -0.0254 | 0

S_h matrix (rows = H, columns = h); '-' where that H has no such horizon
            h=0       h=1       h=2       h=3       h=4       h=5       h=6       h=7
H= 1    -0.0315         -         -         -         -         -         -         -
H= 2    -0.1456    0.0448         -         -         -         -         -         -
H= 4     0.1910   -0.0628   -0.0989   -0.0110         -         -         -         -
H= 8    -0.0254   -0.2897   -0.2288   -0.2032   -0.2276   -0.2448   -0.3388   -0.2362

self-consistency check: H=1 == B1 exactly (0.575204) -> asserted


### 结果：$H$ 不是问题，这份数据也回答不了 $H$

| $H$ | params | overall val MSE | mse @ $h=0$ | $S_0$ | 有效视界 |
|---:|---:|---:|---:|---:|---:|
| 1 | 23,048 | 0.575204 | 0.575204 | −0.0315 | 0 |
| 2 | 24,080 | 0.602890 | 0.638865 | −0.1456 | 0 |
| 4 | 26,144 | 0.613326 | **0.451150** | **+0.1910** | **1** |
| 8 | 30,272 | 0.853913 | 0.571828 | −0.0254 | 0 |

$S_h$ 矩阵（行 = $H$，列 = $h$）：

| | h=0 | h=1 | h=2 | h=3 | h=4 | h=5 | h=6 | h=7 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| $H=1$ | −0.0315 | | | | | | | |
| $H=2$ | −0.1456 | +0.0448 | | | | | | |
| $H=4$ | **+0.1910** | −0.0628 | −0.0989 | −0.0110 | | | | |
| $H=8$ | −0.0254 | −0.2897 | −0.2288 | −0.2032 | −0.2276 | −0.2448 | −0.3388 | −0.2362 |

**四条读法**

1. **自我一致性通过。** $H=1$ 的 `0.575204` 与 B1 **逐位相同**（代码里 `assert` 住了）——同 hidden、同 seed、同样本，两条独立代码路径给出同一个数。
2. **$S_0$ 是非单调的**：−0.03 → −0.15 → **+0.19** → −0.03。如果 $H$ 有系统性影响，$S_0$ 应该随 $H$ 平滑变化。相邻 $H$ 之间从 −0.15 跳到 +0.19，在 1 条轨迹、69 个高度重叠的样本上，这**正是噪声的样子**。
3. **那个唯一的正格（$H=4$, $h=0$）不能用。** 它是 $4\times8$ 比较网格里的**一个格子**；顺着最佳格子去选 $H$，就是在做多重比较。$H=4$ 在 $h=1,2,3$ 上全部为负，说明它并没有"更好"，只是那一格恰好偏正。
4. **`overall val MSE` 随 $H$ 单调上升**（0.575 → 0.603 → 0.613 → 0.854）——但这**不能**读成"$H=8$ 是最差的模型"，因为目标变长了、任务变难了。这是 §6 那条"不同任务不可比"的同一个规则。

**结论**：

> $h=0$ 上的失败**与 $H$ 无关**（$H=1$ 也失败）。所以这不是"$H$ 选错了"的问题——**本课因此可以把 $H$ 从怀疑名单上划掉**：限制在数据覆盖，不在 $H$。

这是一个**否定性的收获**，但很有价值：它把"要不要调 $H$"这个问题关掉了，并把注意力推回到唯一真正的瓶颈上。

## 10. $K$ 越大为什么越不越界？

§8 留下一个未被解释的观察：裁剪率随 $K$ 单调下降（`0.947 → 0.121`），而成功率恒为 0。当时写下的**假设**是：

> $K$ 大时策略执行 chunk 中较远的预测，而那些预测更接近训练集的动作分布（$h$ 大的目标更"平均"），所以更少越界。

这个假设**有一半不需要闭环就能测**：如果成立，模型在 $h$ 上的预测幅度应该**随 $h$ 减小**。先做离线检查。

In [10]:
with torch.no_grad():
    pred_val = model(torch.from_numpy(X_val).to(device)).cpu().numpy()   # (N, H, 8)

pred_mag = np.abs(pred_val).mean(axis=(0, 2))     # mean |a| of the prediction, per h
targ_mag = np.abs(Y_val).mean(axis=(0, 2))        # mean |a| of the expert target, per h

print("normalised action space; |a| = mean absolute value over samples x action dims")
print(f"{'h':>2} | {'mean|pred|':>10} | {'mean|target|':>12} | pred/target")
print("-" * 48)
for h in range(H):
    print(f"{h:>2} | {pred_mag[h]:>10.4f} | {targ_mag[h]:>12.4f} | "
          f"{pred_mag[h] / targ_mag[h]:>11.3f}")

print()
print("If the hypothesis were right, mean|pred| would DECREASE with h.")

normalised action space; |a| = mean absolute value over samples x action dims
 h | mean|pred| | mean|target| | pred/target
------------------------------------------------
 0 |     0.8256 |       0.5891 |       1.402
 1 |     0.8424 |       0.6034 |       1.396
 2 |     0.8618 |       0.6182 |       1.394
 3 |     0.9213 |       0.6335 |       1.454
 4 |     0.9233 |       0.6492 |       1.422
 5 |     0.9647 |       0.6653 |       1.450
 6 |     0.9670 |       0.6818 |       1.418
 7 |     0.9540 |       0.6986 |       1.365

If the hypothesis were right, mean|pred| would DECREASE with h.


In [11]:
import gymnasium as gym
import mani_skill.envs

MAX_STEPS = 200


def as_numpy(v):
    if isinstance(v, torch.Tensor):
        return v.detach().cpu().numpy()
    return np.asarray(v)


@torch.no_grad()
def rollout_chunk_policy(seed, K, max_steps=MAX_STEPS):
    # Predict H actions, execute the first K, re-observe, repeat.
    assert 1 <= K <= H, (K, H)

    env = gym.make("PickCube-v1", obs_mode="state", control_mode=CONTROL_MODE, num_envs=1)
    try:
        # gymnasium's default TimeLimitWrapper caps episodes at 50 steps, which
        # silently truncated every earlier rollout of this notebook (Lesson 2 hit
        # exactly this). Override it AND assert, so the cap can never explain a
        # result again.
        env._max_episode_steps = max_steps
        assert env._max_episode_steps == max_steps, env._max_episode_steps

        obs, _ = env.reset(seed=seed)
        low = np.asarray(env.action_space.low).reshape(-1)
        high = np.asarray(env.action_space.high).reshape(-1)
        assert low.shape == high.shape == (ACTION_DIM,)

        steps = replans = clipped = 0
        success = done = False
        model.eval()

        # Per-executed-step record, so the clipping trend can be attributed.
        executed = []            # (chunk position h, mean|a| of the raw command, clipped?)
        prev, d_sum, d_n = None, 0.0, 0

        while steps < max_steps and not done:
            state = as_numpy(obs).reshape(-1).astype(np.float32)
            assert state.shape == (42,), state.shape
            x = torch.from_numpy(
                ((state - state_mean) / state_std).astype(np.float32)
            ).unsqueeze(0).to(device)

            chunk = model(x)[0].cpu().numpy() * action_std + action_mean   # de-normalise
            replans += 1

            for h_idx, predicted_action in enumerate(chunk[:K]):   # only the first K
                action = np.clip(predicted_action, low, high)
                is_clipped = bool(np.any(np.abs(action - predicted_action) > 1e-6))
                clipped += int(is_clipped)
                executed.append((h_idx, float(np.abs(predicted_action).mean()), is_clipped))
                if prev is not None:
                    d_sum += float(np.abs(predicted_action - prev).mean())
                    d_n += 1
                prev = predicted_action

                obs, reward, terminated, truncated, info = env.step(action.astype(np.float32))
                steps += 1
                if "success" in info:
                    success |= bool(as_numpy(info["success"]).reshape(-1)[0])
                done = (bool(as_numpy(terminated).reshape(-1)[0])
                        or bool(as_numpy(truncated).reshape(-1)[0]))
                if done or steps >= max_steps:
                    break

        success |= bool(as_numpy(env.unwrapped.evaluate()["success"]).reshape(-1)[0])
        return dict(seed=seed, K=K, success=success, steps=steps, replans=replans,
                    clipped_fraction=clipped / max(steps, 1),
                    open_loop_seconds=K / CONTROL_HZ,
                    mean_abs_a=float(np.mean([m for _, m, _ in executed])),
                    mean_delta_a=d_sum / max(d_n, 1),
                    executed=executed)
    finally:
        env.close()


K_VALUES = (1, 2, 4, 8)
SEEDS = (100, 101, 102, 103, 104)

results = [rollout_chunk_policy(seed=s, K=k) for k in K_VALUES for s in SEEDS]

print(f"{'K':>2} | {'success':>7} | {'clip frac':>9} | {'mean|a|':>8} | "
      f"{'mean|da|':>9} | replans")
print("-" * 62)
for k in K_VALUES:
    rs = [r for r in results if r["K"] == k]
    ok = sum(r["success"] for r in rs)
    print(f"{k:>2} | {str(ok) + '/' + str(len(rs)):>7} | "
          f"{np.mean([r['clipped_fraction'] for r in rs]):>9.3f} | "
          f"{np.mean([r['mean_abs_a'] for r in rs]):>8.3f} | "
          f"{np.mean([r['mean_delta_a'] for r in rs]):>9.4f} | "
          f"{np.mean([r['replans'] for r in rs]):>7.1f}")

print()
print("expert reference (measured in 2.9): mean |da| = 0.0077; random fixture = 0.6723")
print()
print("per (K, chunk position h):  mean|a| / clip fraction")
print("      " + "".join(f"{('h=' + str(h)):>14}" for h in range(H)))
for k in K_VALUES:
    row = []
    for h in range(H):
        recs = [(m_, c_) for r in results if r["K"] == k
                for (hh, m_, c_) in r["executed"] if hh == h]
        row.append("     -        " if not recs
                   else f"{np.mean([m_ for m_, _ in recs]):>6.2f}/"
                        f"{np.mean([c_ for _, c_ in recs]):>5.2f}")
    print(f"K={k:>2} " + "".join(f"{x:>14}" for x in row))


2026-09-24 11:01:07,460 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
[2026-09-24 11:01:07.556] [svulkan2] [error] GLFW error: Linux: Failed to query input device: Permission denied. You may suppress this message by unsetting environment variable DISPLAY so SAPIEN will not attempt to test on-screen rendering


2026-09-24 11:01:08,123 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


[2026-09-24 11:01:08.214] [svulkan2] [error] GLFW error: Linux: Failed to query input device: Permission denied. You may suppress this message by unsetting environment variable DISPLAY so SAPIEN will not attempt to test on-screen rendering


2026-09-24 11:01:08,683 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


...

2026-09-24 11:01:17,431 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


[2026-09-24 11:01:17.518] [svulkan2] [error] GLFW error: Linux: Failed to query input device: Permission denied. You may suppress this message by unsetting environment variable DISPLAY so SAPIEN will not attempt to test on-screen rendering


2026-09-24 11:01:17,943 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


[2026-09-24 11:01:18.035] [svulkan2] [error] GLFW error: Linux: Failed to query input device: Permission denied. You may suppress this message by unsetting environment variable DISPLAY so SAPIEN will not attempt to test on-screen rendering


 K | success | clip frac |  mean|a| |  mean|da| | replans
--------------------------------------------------------------
 1 |     0/5 |     0.947 |    3.224 |    1.5156 |   200.0
 2 |     0/5 |     0.370 |    1.365 |    0.4109 |   100.0
 4 |     0/5 |     0.200 |    0.799 |    0.0346 |    50.0
 8 |     0/5 |     0.121 |    0.786 |    0.0367 |    25.0

expert reference (measured in 2.9): mean |da| = 0.0077; random fixture = 0.6723

per (K, chunk position h):  mean|a| / clip fraction
                 h=0           h=1           h=2           h=3           h=4           h=5           h=6           h=7
K= 1     3.22/ 0.95     -             -             -             -             -             -             -        
K= 2     1.33/ 0.39    1.40/ 0.35     -             -             -             -             -             -        
K= 4     0.80/ 0.11    0.79/ 0.10    0.80/ 0.28    0.81/ 0.30     -             -             -             -        
K= 8     0.78/ 0.16    0.79/ 0.12    0.8

### 实测结果

**离线（$h=0$ 是唯一公平的比较量）**

| model | params | val MSE @ $h=0$ | overall val MSE |
|---|---:|---:|---:|
| mean-action baseline（预测 0） | 0 | **0.557645** | — |
| B1 single-step hidden=128 | 23,048 | 0.575204 | 0.575204 |
| B2 single-step hidden=150（参数匹配） | 30,308 | **0.427904** | 0.427904 |
| chunked $H=8$ hidden=128 | 30,272 | 0.571828 | 0.853913 |

per-horizon（chunked vs 它自己的 baseline）：

| $h$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| chunked | 0.5718 | 0.7654 | 0.7756 | 0.8067 | 0.8730 | 0.9372 | 1.0650 | 1.0367 |
| baseline | 0.5576 | 0.5935 | 0.6312 | 0.6705 | 0.7111 | 0.7529 | 0.7955 | 0.8386 |
| better? | no | no | no | no | no | no | no | no |

$$\boxed{\text{有效视界}=0}$$

**在线（$K$ 扫描，5 个 seed）**

| $K$ | 开环区间 | success | mean steps | mean replans | mean clipped |
|---:|---:|---:|---:|---:|---:|
| 1 | 0.05 s | **0/5** | 200.0 | 200.0 | **0.947** |
| 2 | 0.10 s | **0/5** | 200.0 | 100.0 | 0.370 |
| 4 | 0.20 s | **0/5** | 200.0 | 50.0 | 0.200 |
| 8 | 0.40 s | **0/5** | 200.0 | 25.0 | **0.121** |

### 这些数字在说什么

1. **有效视界 = 0。** chunked 模型在**每一个** $h$ 上都不如"预测训练集平均动作"。也就是说在这份数据上，$H=8$ 完全没有被支持——不是"远的那几步差"，而是**连第 1 步都没有信息**。
2. **chunked@$h=0$（0.5718）≈ B1@$h=0$（0.5752）。** 多步目标对"预测下一步"这个能力**既没有帮助也没有伤害**。所以本课**没有**得到"chunking 提升了预测"的证据。
3. **唯一优于 baseline 的是 B2**（0.4279 vs 0.5576，好 23%），而 B1 反而**差于** baseline。两者只差 hidden（128 vs 150）——所以这个差别是**容量**，不是 chunking。而 $h=0$ 与 B2 的差距用的是 1 条轨迹的 69 个样本，未必是真实效应（见自检第 7 题）。
4. **$K$ 强烈改变动作饱和，但不改变成功率。** 裁剪率从 `0.947`（$K=1$）单调降到 `0.121`（$K=8$），而 success 在四种 $K$ 下全是 `0/5`。这正是本课预设的落点：$K$ 的产物是**结构量**（重规划次数 200 → 25），不是任务结果。
5. **所有行都跑满 200 步**，所以那个 TimeLimit 覆写确实在起作用——不修的话四行都会写成 `50`。
6. **chunked 严重过拟合**：train loss 掉到 `0.0096`，val 停在 `0.854`。这与 Lesson 2 是同一个失败模式（4 条训练 episode，1 条验证）。

### 诚实的边界

每个 $K$ 只有 5 个 seed 且全失败，所以成功率的单侧 95% 上界仍然约 **45%**（3.3 里的 Clopper-Pearson 公式，$K=5$）。因此本课**不能**区分：

- "chunking 对闭环没用"，还是
- "在 5 条 episode 上任何策略都没用"。

**能站出来的是离线那半部分，而它是负面的。** 这与 Lesson 2 的结论一致：目前限制不是架构选择，而是**数据覆盖**。chunking 是否有效，要等数据够多之后才能回答——本课能确定的是**测量方法已经就位**（正确的样本、受控的对比、可断言的恒等式、修好的时间上限）。

### 一个意料之外但可解释的观察

$K$ 越大，**动作越不容易越界**。一个可能的解释是：$K=1$ 时策略每一步重新预测，而一个过拟合的策略在自己的分布漂移上反复取样，容易推出极端值；$K$ 较大时它更多在执行 chunk 中**较远的**那些预测，而那些预测更接近训练集的动作分布（因为 $h$ 大的目标本身更"平均"）。这只是**假设**，本课没有设计实验去区分它——如果要验证，需要记录每个 $K$ 下**被执行动作的分布**，而不只是裁剪率。

### §10 的结果：假设被否证，换成一个有证据的解释

**离线：假设的前半部分是错的。**

| $h$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| `mean abs(pred)` | 0.826 | 0.842 | 0.862 | 0.921 | 0.923 | 0.965 | 0.967 | 0.954 |
| `mean abs(target)` | 0.589 | 0.603 | 0.618 | 0.634 | 0.649 | 0.665 | 0.682 | 0.699 |
| `pred/target` | 1.40 | 1.40 | 1.39 | 1.45 | 1.42 | 1.45 | 1.42 | 1.37 |

模型在 $h$ 上的预测幅度**随 $h$ 增加**（0.826 → 0.954），目标幅度也增加。所以"越远的预测越接近均值"是**错的**。

但要记下另一件事：**每一行的 `pred/target ≈ 1.4`**——模型系统性地把动作幅度**放大约 40%**。这与它大量越界是自洽的，而且这是一个关于**模型**的发现，不是关于 $K$ 的。

**在线：决定性的一组数。**

| $K$ | success | 裁剪率 | `mean abs(a)` | **`mean abs(Δa)`** | 重规划 |
|---:|---:|---:|---:|---:|---:|
| 1 | 0/5 | 0.947 | **3.224** | **1.5156** | 200 |
| 2 | 0/5 | 0.370 | 1.365 | 0.4109 | 100 |
| 4 | 0/5 | 0.200 | 0.799 | **0.0346** | 50 |
| 8 | 0/5 | 0.121 | 0.786 | 0.0367 | 25 |

按 $(K, h)$ 拆开——**权重完全相同，样本位置完全相同**：

| | h=0 | h=1 | h=2 | h=3 | h=4 | h=5 | h=6 | h=7 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| $K=1$ | **3.22 / 0.95** | | | | | | | |
| $K=2$ | 1.33 / 0.39 | 1.40 / 0.35 | | | | | | |
| $K=4$ | 0.80 / 0.11 | 0.79 / 0.10 | 0.80 / 0.28 | 0.81 / 0.30 | | | | |
| $K=8$ | 0.78 / 0.16 | 0.79 / 0.12 | 0.80 / 0.11 | 0.79 / 0.14 | 0.78 / 0.11 | 0.79 / 0.10 | 0.79 / 0.13 | 0.77 / 0.10 |

（每格 = `mean abs(a) / 裁剪率`）

**三张表合起来说的是：极端动作不是 chunk 位置的属性，而是状态分布的属性。**

- 在 $K=8$ 内部，$h=0$ 与 $h=7$ 的 `mean abs(a)` 几乎相同（0.78 vs 0.77）——**位置不重要**；
- 但**同一个 $h=0$**，$K=1$ 时是 **3.22**，$K=8$ 时是 **0.78**——**差 4 倍**。

权重相同、样本位置相同、同一个 checkpoint，唯一变化的是**策略自己走到了哪些状态**。

所以修正后的解释是：

> $K=1$ 每步重规划，相邻步的动作剧烈跳变（`mean abs(Δa) = 1.52`，**比随机动作 fixture 的 0.6723 还差**），把机器人推到自己从未见过的状态上；在这些状态里连 $h=0$ 的预测都严重外推（幅度放大 40%），于是被裁剪。$K$ 增大后，一个 chunk 内的动作**内部一致**（`mean abs(Δa)` 掉到 `0.035`，与专家的 `0.0077` 同一量级），状态演化平稳，预测幅度回到正常范围，裁剪随之减少。

**也就是说：这里 chunking 真正起作用的地方是 temporal consistency（时间一致性），不是"看得更远"。** 这是 chunking 的标准论据之一，本课意外地把它测了出来——尽管 success 仍然是 `0/5`。

> **边界**：$K=1$ 与 $K=8$ 看的是**不同的状态序列**，所以那个"4 倍"是**状态分布差异**与 **chunk 内部一致性**的合成，本课没有设计实验把两者分开。
>
> **能把它们分开的下一个测量**（本课没做）：比较 (a) 单个 chunk 内部相邻预测的平滑度 `mean abs(chunk[h+1] - chunk[h])`，与 (b) 在闭环记录下来的状态序列上**重规划**得到的 $h=0$ 预测之间的跳变。前者是纯"chunk 内部一致性"，后者是纯"重规划抖动"。
>
> 另外注意 $K=4$ 与 $K=8$ 的 `mean abs(Δa)` 几乎相同（0.0346 vs 0.0367），说明稳定性在大约 $K=4$ 就已经饱和——但 success 依然是 0，所以这个"稳定"并没有变成任务能力。

### $K$ 扫描报告的是什么

$K$ 是**执行时**策略，所以同一份权重可以扫出一张表：

| 报告字段 | 它在回答什么 |
|---|---|
| `success` | 任务结果（`env.unwrapped.evaluate()`） |
| `steps` | 实际步数（**必须**先修掉 50 步上限，否则所有行都是 50） |
| **`replans` $=\lceil\text{steps}/K\rceil$** | **$K$ 真正改变的那个量：决策次数** |
| **`open_loop_seconds` $=K/20$** | 一次预测要盲走多久 |
| `clipped_fraction` | 动作越界比例（Lesson 2 是 `0.995`；本课 $K=1$ 的旧记录是 `46/50 = 0.92`） |

**实测结果**（见上一节）：四种 $K$ 的 success 全是 `0/5`，所以本课的产物确实是"决策次数 / 开环区间 / 裁剪率"这组**结构量**，而不是成功率。这不是本课的失败，但它意味着**本课没有也不能证明 chunking 对闭环有用**。

一个漂亮的观察：**$K=1$ 时 chunked 模型被当成单步模型用**，所以 $K=1$ 就是"chunking 退化成单步"的对照组。

## 小结

**核心命题**（要能自己复述）：

> Chunking 把**高频闭环**换成**低频闭环 + 局部的开环承诺**。它减少的是**重新观察的次数**，不是单次误差 $\varepsilon$。

**三个超参数的分工**：

| 符号 | 决定 | 属于 | 本课 |
|---|---|---|---|
| $L$ | 输入多长 | 训练时 | 1（隔离变量） |
| $H$ | 目标多长、丢多少起点 | 训练时 | 8 |
| $K$ | 用掉预测的前几行 | **执行时** | 扫 $\{1,2,4,8\}$ |

**测出来的 vs 还不知道的**：

| 项 | 状态 |
|---|---|
| chunk 样本 325 个、无跨集；naive 会跨 28 个 | 已断言 |
| $H=8$ 比 $H=1$ 多 31.3% 参数（30,272 vs 23,048） | 已实测 |
| `baseline_h` 从 0.5576 涨到 0.8386 | 已实测 |
| per-horizon 曲线与**有效视界** | 已测出：**有效视界 = 0**（每个 $h$ 都不如 baseline） |
| 受控对比（$h=0$，含参数匹配的 B2） | 已测出：chunked `0.5718` = B1 `0.5752`；只有参数更多的 B2 `0.4279` 优于 baseline `0.5576` |
| $K$ 对成功率 / 裁剪率 / 决策次数的影响 | 已测出：success 全为 `0/5`；裁剪率 `0.947 -> 0.121`（$K=1\to8$）；决策次数 `200 -> 25` |
| $H=8$ 是否最优 | **已测**（§9）：$S_0$ 非单调（−0.03 / −0.15 / +0.19 / −0.03），$H$ 不是瓶颈；$H=1$ 也失败 |
| $K$ 为何降低越界 | **已测**（§10）：不是"看得更远"，而是 chunk 内部一致（`mean abs(Δa)` 从 1.52 降到 0.035）；模型幅度系统性放大 40% |
| temporal ensembling 是否必要 | **本课不实现** |

**一句话记住可比较性**：单步模型的 MSE 与 chunked 模型的**总** MSE **不可比**（8 个数 vs 64 个数，后者天生更难）；唯一公平的量是 **$h=0$**。

## 自检

1. Chunking 减少的是**决策次数**还是**单次误差**？为什么这个区分决定它能否"治好" compounding error？
2. 为什么 $K$ 可以训练后扫，而 $H$ 不行？chunk 构造的代码里为什么找不到 $K$？
3. $H=8$ 比 $H=1$ 多 31.3% 参数。为什么这会让"chunked 整体 MSE 更低"不能说明 chunking 有效？B2 是用来干什么的？
4. 为什么单步模型的 MSE 和 chunked 模型的**总** MSE **不能**直接比？公平的比较量是哪一个？为什么？
5. per-horizon MSE 的算术平均**为什么**等于整体验证 loss？如果改用 padding+mask，这个等式还成立吗？
6. `baseline_h` 从 0.5576 涨到 0.8386。这是"越远越难"吗？还有什么在同时变化？有效视界怎么算？
7. 你的 val 只有 1 条轨迹、69 个 chunk 样本。如果验证 MSE 差 0.01，你能说 chunked 更好吗？
8. Chunk 内部是开环的。举一个机械臂上的例子，说明"chunk 内假设了错误的未来状态"会怎样。

9. §9 里 $S_0$ 随 $H$ 的变化是 −0.03 → −0.15 → **+0.19** → −0.03。这个形状本身说明了什么？为什么不能据此选 $H=4$？
10. §10 里 $K=8$ 内部 $h=0$ 与 $h=7$ 的 `mean abs(a)` 几乎相同，但 $K=1$ 的 $h=0$ 是 $K=8$ 的 $h=0$ 的 4 倍。这两件事分别排除了什么、指向了什么？

## 自检：答案

（学习者自己的回答；`> 补正` 是批注。）

**1. Chunking 减少什么？**

> 当 $K>1$ 时，减少的是策略重新规划的次数：执行 $T$ 个控制步，大约调用 $\lceil T/K\rceil$ 次模型。机器人仍每步收到一个动作。Chunking 不保证单步预测误差 $\varepsilon$ 变小；在 chunk 内，状态偏离预期后，后续动作仍按旧观察生成。因此它可能改变误差累积的过程，不能声称"治好" compounding error。$H=8,K=1$ 甚至仍每步重新规划。

> **补正**：$\lceil T/K\rceil$ 就是实测表里的 `mean replans` 一列（$200/1,200/2,200/4,200/8 = 200,100,50,25$），公式与数据完全对上。最后那句是本课最锋利的观察：既然 $H=8,K=1$ 仍然每步重规划，那么"chunking 的收益"就**全部由 $K$ 决定，与 $H$ 无关**——$K=1$ 正是"chunking 退化成单步"的对照组。

**2. 为什么能扫 $K$，却不能直接改 $H$？**

> 训练标签形状是 $[H,8]$，网络最后一层输出 $H\times8$ 个数，所以改 $H$ 就改变标签和网络参数形状，需要构造新样本并重训。$K$ 只决定推理后执行 `chunk[:K]` 中的几步，再重新规划；它不参与监督标签和 loss，所以构造 chunk 的代码里没有 $K$。同一个 $H=8$ checkpoint 可以评估 $K=1,2,4,8$。

> **补正**：完全正确。补一条边界：$K\le H$ 是硬约束（不能执行比预测更多的步），代码里由 `assert 1 <= K <= H` 固定。也正因为 $H$ 与 $K$ 分开，$H-K$ 才是相邻两次预测的重叠长度。

**3. 多 31.3% 参数为什么是混杂因素？B2 做什么？**

> 你的 $42\to128\to128\to8$ 约有 23,048 个参数，$42\to128\to128\to64$ 约有 30,272 个，多 31.3%。若 chunk 模型表现更好，可能是动作块目标的作用，也可能部分来自模型容量增加。
>
> 如果你们实验中的 B2 指"参数量匹配的单步模型"，它就是容量对照：仍预测 $[8]$，但加宽隐藏层，使参数量接近 30,272，再与 chunk 模型比较。例如隐藏宽度 150 时约有 30,308 个参数。如果你的 B2 在 notebook 中另有定义，以那里的代码为准；当前对话没有给出其原始定义。

> **补正**：待确认的部分已确认——本 notebook 里 B2 **就是**参数量匹配的单步模型（`hidden=150`，30,308 个参数），与你的推测一致。"以代码为准"这个保留态度是对的。
>
> 更要紧的是实测结果比设计更有意思：B1（23,048）**差于** baseline，B2（30,308）**优于** baseline。两者只差 hidden 宽度，所以那段差距是**容量**，不是 chunking。而 chunked（30,272）在 $h=0$ 上与 B1（23,048）几乎相同——**多带了 31% 参数却没有换来更好的单步预测**。

**4. 为什么不能比两个模型的总 MSE？**

> 单步模型的 MSE 只考察 $a_t$；chunk 的总 MSE 同时考察 $a_t,\ldots,a_{t+7}$。它们回答的是不同问题。比较当前动作预测能力时，应在相同验证起点、相同归一化和划分下比较 $\mathrm{MSE}_{\text{single}}(\hat a_t,a_t)$ 与 $\mathrm{MSE}_{\text{chunk},h=0}(\hat a_t,a_t)$。再加入 B2，就能进一步检查参数量的影响。chunk 的各个 $h$ 和总 MSE 则用于研究未来预测能力。

> **补正**：不需要补正，这个表述比 notebook 正文更准确（它明确点出了"同验证起点、同归一化、同划分"三个条件）。

**5. 为什么 per-horizon 的平均等于总 loss？**

> 你每个样本都恰有 8 个有效未来动作，每个 $h$ 都有相同数量的样本和 8 个动作维度，而且 `F.mse_loss` 对所有元素等权平均。因此 $L_{\text{total}}=\frac{1}{8}\sum_{h=0}^{7}L_h$。改成 padding + mask 后，较远的 $h$ 通常有更少有效样本。此时总 masked loss 是按各 $h$ 的有效元素数加权平均，一般不再等于八个 $L_h$ 的简单算术平均。

> **补正**：正确。再补一个条件：这个恒等式还要求 loss 是**逐元素平均**；如果写成 `reduction='sum'`，等式会变成 $L_{\text{total}}=\sum_h L_h$。所以恒等式成立需要三个条件同时满足——每个 $h$ 的有效元素数相同、loss 为等权平均、$L_h$ 的定义同为逐元素平均。

**6. `baseline_h` 上升说明越远越难吗？有效视界怎么算？**

> 单看 0.5576→0.8386 不能证明。随着 $h$ 增大，标签从 $a_t$ 换成 $a_{t+h}$；窗口覆盖到的任务阶段、动作幅度和分布也在变化。即使模型预测能力完全不变，baseline 误差也可能变化。
>
> 应逐 $h$ 比较模型与同一 $h$ 的 baseline：$S_h=1-\mathrm{MSE}_{\text{model},h}/\mathrm{MSE}_{\text{baseline},h}$。$S_h>0$ 表示该步优于 baseline。一个明确的"有效视界"定义是：从 $h=0$ 起，连续保持 $S_h>0$ 的步数；不要因为某个较远的孤立步偶然转正，就跳过中间失败的步。若连续有效 $m$ 步、控制频率 20 Hz，则覆盖约 $m/20$ 秒。阈值也可以设得比零更严格，但必须事先固定。

> **补正**：**这一条比 notebook 原来的定义更好，已按你的版本改写。** 原定义是 $\max\{h:\text{mse}_h<\text{baseline}_h\}+1$，缺陷正是你指出的那个：某个远处的孤立步偶然转正会把中间所有失败步一起算进去。现在代码用"从 $h=0$ 起连续"的版本，并把阈值固定为常量 `USEFUL_THRESHOLD`。
>
> 你问的"代码里的 `baseline_h` 具体预测什么"——它是 `(Y_val ** 2).mean(axis=(0, 2))`，其中 `Y_val` 已用**训练集**统计量归一化。所以在归一化空间里它预测的是 **0**，也就是**训练集平均动作**；而评估落在**验证集**样本上。因此它是"用训练集均值去预测验证集"，与 Lesson 2 的 mean-action baseline 是同一个量，只是空间不同。
>
> 按你的定义重算：$S_0 = 1-0.571828/0.557645 = -0.0254 < 0$，所以**有效视界仍是 0**，结论不变。

**7. 一个 episode、69 个 chunk，相差 0.01 能说更好吗？**

> 不能。69 个 chunk 高度重叠，不是 69 次独立试验；0.01 也可能来自这一条轨迹的特点、训练随机种子或 checkpoint 选择。至少要在多个未见 episode 上做配对比较，汇报每条 episode 的结果及整体分布；闭环还要比较相同 seeds 下的成功率。先核对上面提到的 67/69 划分差异。

> **补正**：正确，而且"高度重叠 ⇒ 不是 69 次独立试验"正是有效样本量远小于 69 的原因。
>
> 你记的 67/69 不是 bug，是两个不同的量：验证 episode 有 **76 帧**原始状态，去掉尾部 $H-1=7$ 个起点后得到 **69** 个 chunk 样本（$76-7=69$）。所以 `0.6939`（69 个 chunk 样本）与 `0.7874`（76 帧）本就不该相等——后者是单步视角的全帧均值，前者是丢尾巴后的 chunk 起点均值。67 是另外两条 $T=74$ 的 episode（$74-7=67$）。
>
> 再补一条：**"有效视界 = 0"同样只是这一条轨迹上的陈述。** 它是个否定性结论（比肯定性结论稳健），但仍不能升级为分布性结论；而且本课报的 0.5718 vs 0.5752 之差（0.003）远在任何可信噪声底之下，正确说法是"测不出差异"，不是"两者相等"。

**8. Chunk 内错误未来状态的例子**

> 机器人根据当前状态预测"靠近方块→闭合夹爪→抬起→搬运"。实际执行时，夹爪在第二步没夹住方块，但 $K=8$ 会继续执行预先生成的抬起和搬运动作：机械臂走了，方块还留在桌上。较小的 $K$ 能更早利用新观察发现偏差并重新规划；但策略能否真正恢复，还取决于它是否学过这类失败状态和恢复动作。

> **补正**：不需要补正。这个例子精确对应"chunk 内嵌了一个对未来的**隐含预测**；实际状态一旦偏离，动作序列就是过期的"——也就是 **chunk 内部的状态分布漂移**，是一段被局部化的 compounding error。
>
> 一个可以补的观察：本课唯一记录到的失败信号是 `clipped_fraction`（$K=1$ 时 0.947，$K=8$ 时 0.121），它太粗，看不见你描述的这种"动作合法但基于错误状态"的失败。要测它需要记录**被执行动作的分布**与**任务进度**，这也正是"$K$ 越大越不越界"那个假设目前无法验证的原因。
>
> 至于"能否恢复"——那取决于策略是否学过失败状态与恢复动作，正是本项目 P2 的方向（failure detection and recovery），不是 chunking 本身能解决的。

---

**第 9、10 题答案待补。** 这两题来自 §9 / §10 新增的测量，题目本身已经把线索写在里面了。